In [1]:
import os
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

seed = 42
set_seed(seed)

/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. LFF v2

## Embedding Cosine similarity

In [2]:
base_output_path = "./output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test.jsonl"
lff2_output_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v2_test.jsonl"
lff2_similarity_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v2_test_sim.jsonl"

base_output = read_data(base_output_path)
lff2_output = read_data(lff2_output_path)
lff2_similarity = read_data(lff2_similarity_path)

print(f"base_output: {len(base_output)}")
print(f"lff2_output: {len(lff2_output)}")
print(f"lff2_similarity: {len(lff2_similarity)}")

base_output: 1319
lff2_output: 1319
lff2_similarity: 1319


In [3]:
all_similarity = []
correct_similarity = []
incorrect_similarity = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    pred_answer = base_output[i]["pred_ans"]
    similarity = lff2_similarity[i]["max_sim"]

    all_similarity.append(similarity)

    if true_answer == pred_answer:
        correct_similarity.append(similarity)
    else:
        incorrect_similarity.append(similarity)

print(f"all_similarity: {len(all_similarity)}")       
print(f"all_similarity: {np.mean(all_similarity)}")
print(f"correct_similarity: {len(correct_similarity)}")
print(f"correct_similarity: {np.mean(correct_similarity)}")
print(f"incorrect_similarity: {len(incorrect_similarity)}")
print(f"incorrect_similarity: {np.mean(incorrect_similarity)}")

100%|██████████| 1319/1319 [00:00<00:00, 769870.16it/s]

all_similarity: 1319
all_similarity: 0.38671504809514784
correct_similarity: 1074
correct_similarity: 0.38544667277700184
incorrect_similarity: 245
incorrect_similarity: 0.3922751913265306


## Correct - Incorrect

In [4]:
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    base_pred_answer = base_output[i]["pred_ans"]
    lff2_pred_answer = lff2_output[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if lff2_pred_answer == true_answer:
            correct_correct.append(lff2_output[i])
        else:
            correct_incorrect.append(lff2_output[i])
    else:
        if lff2_pred_answer == true_answer:
            incorrect_correct.append(lff2_output[i])
        else:
            incorrect_incorrect.append(lff2_output[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

100%|██████████| 1319/1319 [00:00<00:00, 708994.87it/s]

correct_correct: 939
Index: [0, 1, 3, 6, 9, 10, 11, 14, 15, 17, 18, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 39, 42, 44, 47, 48, 49, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 65, 66, 67, 68, 69, 70, 71, 72, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 88, 90, 91, 92, 93, 94, 95, 96, 97, 99, 100, 101, 103, 105, 106, 108, 109, 112, 113, 115, 117, 118, 120, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 140, 142, 143, 144, 146, 149, 152, 154, 155, 156, 160, 161, 163, 164, 165, 166, 167, 168, 169, 170, 171, 173, 174, 176, 178, 179, 180, 183, 185, 188, 189, 190, 191, 193, 194, 195, 196, 197, 199, 200, 202, 203, 204, 206, 207, 208, 211, 212, 213, 215, 216, 217, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 235, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 262, 263, 264, 265, 266, 268, 269, 270, 274, 275, 276, 277, 278, 280, 281, 282, 284, 285, 286, 287, 288, 289, 290, 291, 

# 2. LFF v3

## Embedding Cosine similarity

In [5]:
base_output_path = "./output/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test.jsonl"
lff3_output_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test.jsonl"
lff3_similarity_path = "./output/GSM8K_Llama-3-8B-Instruct_LFF_v3_test_sim.jsonl"

base_output = read_data(base_output_path)
lff3_output = read_data(lff3_output_path)
lff3_similarity = read_data(lff3_similarity_path)

print(f"base_output: {len(base_output)}")
print(f"lff3_output: {len(lff3_output)}")
print(f"lff3_similarity: {len(lff3_similarity)}")

base_output: 1319
lff3_output: 1319
lff3_similarity: 1319


In [6]:
all_similarity = []
correct_similarity = []
incorrect_similarity = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    pred_answer = base_output[i]["pred_ans"]
    similarity = lff3_similarity[i]["max_sim"]

    all_similarity.append(similarity)

    if true_answer == pred_answer:
        correct_similarity.append(similarity)
    else:
        incorrect_similarity.append(similarity)

print(f"all_similarity: {len(all_similarity)}")       
print(f"all_similarity: {np.mean(all_similarity)}")
print(f"correct_similarity: {len(correct_similarity)}")
print(f"correct_similarity: {np.mean(correct_similarity)}")
print(f"incorrect_similarity: {len(incorrect_similarity)}")
print(f"incorrect_similarity: {np.mean(incorrect_similarity)}")

  0%|          | 0/1319 [00:00<?, ?it/s]

100%|██████████| 1319/1319 [00:00<00:00, 802711.40it/s]

all_similarity: 1319
all_similarity: 0.28476421827615617
correct_similarity: 1074
correct_similarity: 0.2844397404562384
incorrect_similarity: 245
incorrect_similarity: 0.2861866230867347


## Correct - Incorrect

In [7]:
correct_correct = []
correct_incorrect = []
incorrect_correct = []
incorrect_incorrect = []

for i in tqdm(range(len(base_output))):
    true_answer = base_output[i]["answer"]
    base_pred_answer = base_output[i]["pred_ans"]
    lff3_pred_answer = lff3_output[i]["pred_ans"]

    if base_pred_answer == true_answer:
        if lff3_pred_answer == true_answer:
            correct_correct.append(lff3_output[i])
        else:
            correct_incorrect.append(lff3_output[i])
    else:
        if lff3_pred_answer == true_answer:
            incorrect_correct.append(lff3_output[i])
        else:
            incorrect_incorrect.append(lff3_output[i])

print(f"correct_correct: {len(correct_correct)}\nIndex: {[o['index'] for o in correct_correct]}\n")
print(f"correct_incorrect: {len(correct_incorrect)}\nIndex: {[o['index'] for o in correct_incorrect]}\n")
print(f"incorrect_correct: {len(incorrect_correct)}\nIndex: {[o['index'] for o in incorrect_correct]}\n")
print(f"incorrect_incorrect: {len(incorrect_incorrect)}\nIndex: {[o['index'] for o in incorrect_incorrect]}\n")
print(f"total num: {len(correct_correct) + len(correct_incorrect) + len(incorrect_correct) + len(incorrect_incorrect)}")

100%|██████████| 1319/1319 [00:00<00:00, 680394.41it/s]

correct_correct: 948
Index: [0, 1, 3, 6, 9, 10, 11, 14, 15, 17, 18, 22, 23, 24, 25, 26, 27, 29, 31, 32, 33, 34, 35, 36, 39, 40, 41, 42, 43, 44, 47, 48, 49, 51, 53, 54, 55, 56, 58, 59, 61, 65, 66, 67, 68, 69, 70, 71, 72, 76, 77, 78, 79, 80, 81, 82, 83, 84, 86, 88, 90, 91, 92, 93, 95, 97, 99, 100, 101, 103, 105, 106, 108, 109, 110, 112, 113, 114, 115, 116, 117, 118, 120, 121, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 140, 142, 144, 146, 149, 152, 156, 158, 159, 160, 161, 163, 164, 165, 166, 167, 168, 169, 170, 171, 173, 174, 176, 178, 179, 180, 181, 182, 183, 185, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 202, 203, 204, 206, 207, 208, 211, 212, 213, 215, 216, 217, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 235, 237, 238, 239, 240, 241, 243, 246, 247, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 262, 263, 264, 265, 266, 268, 269, 270, 271, 274, 275, 276, 277, 278, 280, 281, 284, 285, 286, 287, 288, 289